# Mini-projeto CMP629 — MountainCar-v0

**Análise reprodutível da triagem (108 execuções) e demonstração de Q-learning, SARSA(λ) e Dyna-Q.**

**Importante:** os dados originais são da triagem, não do experimento confirmatório. A demonstração opcional é independente dos resultados da triagem. Use **Ambiente de execução > Executar tudo** no Google Colab.

In [ ]:
%pip -q install "gymnasium[classic-control]>=1.0,<2" pandas numpy matplotlib seaborn
import io, gzip, pathlib, urllib.request, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import gymnasium as gym
from collections import defaultdict, Counter
from IPython.display import display

OUT = pathlib.Path("figuras_geradas")
OUT.mkdir(exist_ok=True)
ROOT = "https://raw.githubusercontent.com/antoniomoraes85/reinforcement-learning-projects/cmp629-mountaincar-triagem-auditoria-20261011/mountaincar-tabular"
print("Python pronto; Gymnasium", gym.__version__)

## 1. Carregamento e auditoria de integridade

Os 108 registros são resumos *por treinamento* (cada configuração/seed), não JSONs de transições individuais. Ler CSV comprimido preserva as 108 linhas. As métricas da avaliação são separadas das métricas do treino exploratório.

In [ ]:
url = f"{ROOT}/dados/triagem/resumo_execucoes.csv.gz"
with urllib.request.urlopen(url, timeout=30) as resposta:
    dados = resposta.read()
df = pd.read_csv(io.BytesIO(gzip.decompress(dados)))
assert len(df) == 108, f"Esperadas 108 linhas, recebidas {len(df)}"
assert not df.duplicated(['algorithm','discretization','seed','alpha','lambda','epsilon_decay','N']).any()
assert set(df.algorithm) == {"q_learning", "sarsa_lambda", "dyna_q"}
assert set(df.discretization) == {"4x2", "12x4"}
assert set(df.seed) == {100, 1100, 2100}
assert df.evaluation_success_rate.between(0, 1).all()
print("Validação de resumos: 108/108; hashes únicos; três seeds")
display(df.head())

## 2. Seleção global pré-definida

Selecionar a configuração de maior AUC de treinamento **agregando ambas discretizações**. Não selecionar pelo desempenho da avaliação. Os nomes antigos relativos a "estabilidade" são legados da triagem e representam o *primeiro atingimento do limiar*.

In [ ]:
hp_cols = ["algorithm", "alpha", "lambda", "epsilon_decay", "N"]
configs = df.groupby(hp_cols, as_index=False).agg(auc_mean=("auc_normalized","mean"), final_window_success=("final_window_success","mean"))
selecao = configs.sort_values(["algorithm","auc_mean","final_window_success"], ascending=[True,False,False]).drop_duplicates("algorithm")[hp_cols]
selecionados = df.merge(selecao, on=hp_cols, how="inner")
assert len(selecionados) == 18
res = selecionados.groupby(["algorithm","discretization"]).agg(
    n_seeds=("seed","nunique"), auc=("auc_normalized","mean"),
    treino_final=("final_window_success","mean"),
    avaliacao=("evaluation_success_rate","mean"),
    dp_avaliacao=("evaluation_success_rate","std")).reset_index()
display(selecao)
display(res.round(4))

## 3. Figuras para apresentação

Cada barra sintetiza três seeds de treinamento independentes. As barras de erro representam desvio-padrão entre seeds, não intervalo de confiança. Não tratar episódios de avaliação como novas replicações de treino.

In [ ]:
cores = {"4x2":"#E0A44A", "12x4":"#2877A2"}
nomes = {"q_learning":"Q-learning", "sarsa_lambda":"SARSA(λ)", "dyna_q":"Dyna-Q"}
ordem = ["sarsa_lambda","q_learning","dyna_q"]
fig, ax = plt.subplots(figsize=(10,5.5))
x = np.arange(3)
for i, disc in enumerate(["4x2","12x4"]):
    d = res.set_index(["algorithm","discretization"])
    ys = [d.loc[(a,disc),"avaliacao"]*100 for a in ordem]
    es = [d.loc[(a,disc),"dp_avaliacao"]*100 for a in ordem]
    ax.bar(x+(i-.5)*.38, ys, .38, yerr=es, capsize=4, label=disc, color=cores[disc])
ax.set_xticks(x, [nomes[a] for a in ordem]); ax.set_ylim(0,120)
ax.set_ylabel("Sucesso na avaliação (%)"); ax.set_title("MountainCar: avaliação independente por discretização")
ax.legend(title="Discretização"); ax.grid(axis="y", alpha=.25)
fig.tight_layout(); fig.savefig(OUT/"avaliacao_por_discretizacao.png", dpi=220); plt.show()

In [ ]:
fig, axs = plt.subplots(1,2,figsize=(12,4.5))
for ax, col, tit in [(axs[0],"auc","AUC média de treinamento"),(axs[1],"treino_final","Sucesso na janela final de treino")]:
    piv = res.pivot(index="algorithm",columns="discretization",values=col).reindex(ordem)
    sns.heatmap(piv, annot=True, fmt=".3f", vmin=0, vmax=1 if col=="treino_final" else .11,
                cmap="Blues", ax=ax, cbar=True)
    ax.set_yticklabels([nomes[a] for a in ordem], rotation=0)
    ax.set_title(tit); ax.set_xlabel("Discretização"); ax.set_ylabel("")
fig.tight_layout(); fig.savefig(OUT/"auc_treino_discretizacao.png",dpi=220); plt.show()

In [ ]:
dyna = df[df.algorithm=="dyna_q"].groupby(["N","discretization"],as_index=False).agg(
    sucesso_avaliacao=("evaluation_success_rate","mean"), auc=("auc_normalized","mean"))
fig, axs = plt.subplots(1,2,figsize=(11,4))
for i, metrica in enumerate(["sucesso_avaliacao","auc"]):
    for disc in ["4x2","12x4"]:
        sub = dyna[dyna.discretization==disc]
        axs[i].plot(sub.N, sub[metrica], marker="o", label=disc, color=cores[disc])
    axs[i].set_title("Dyna-Q — "+metrica.replace("_"," "))
    axs[i].set_xlabel("N: simulações por interação"); axs[i].grid(alpha=.25); axs[i].legend()
fig.tight_layout(); fig.savefig(OUT/"sensibilidade_dyna_N.png",dpi=220); plt.show()

## 4. Demonstração **opcional** dos algoritmos

A rotina a seguir implementa novamente uma versão tabular reduzida e **independente** para fins didáticos. Seu resultado NÃO integra os 108 treinamentos documentados. Para uma apresentação rápida execute 40 episódios por algoritmo; não compare seus números com a triagem de 1.000 episódios.

In [ ]:
def discretizador(observacao, resolucao="12x4"):
    bordas = {"4x2": ([-.6,-.2,.2],[0.0]),
              "12x4": ([-1.05,-.9,-.75,-.6,-.45,-.3,-.15,0,.15,.3,.45],[-.02,0,.02])}
    ep, ev = bordas[resolucao]
    return int(np.digitize(observacao[0],ep)), int(np.digitize(observacao[1],ev))

def escolher(q, estado, epsilon, rng):
    if rng.random()<epsilon: return int(rng.integers(3))
    valores=q[estado]; melhores=np.flatnonzero(valores==valores.max())
    return int(rng.choice(melhores))

def treinar_demo(algoritmo, episodios=40, resolucao="12x4", seed=100):
    """Agente de demonstração. NÃO reproduz os 108 treinamentos originais."""
    assert algoritmo in {"q_learning","sarsa_lambda","dyna_q"}
    shape={"4x2":(4,2,3),"12x4":(12,4,3)}[resolucao]
    q=np.zeros(shape); rng=np.random.default_rng(seed); modelo=defaultdict(Counter)
    env=gym.make("MountainCar-v0")
    alpha={"q_learning":.1,"sarsa_lambda":.02,"dyna_q":.05}[algoritmo]
    decay={"q_learning":.99,"sarsa_lambda":.995,"dyna_q":.995}[algoritmo]
    lam=.9; gamma=.99; N=5; epsilon=1.0; historico=[]
    try:
        assert env.action_space.n==3 and env.spec.max_episode_steps==200
        for ep in range(episodios):
            obs,_=env.reset(seed=seed+ep); s=discretizador(obs,resolucao)
            traces=np.zeros_like(q); ac=escolher(q,s,epsilon,rng)
            retorno=0; venceu=False
            for t in range(200):
                if algoritmo!="sarsa_lambda": ac=escolher(q,s,epsilon,rng)
                nova, recompensa, terminou, truncou, _=env.step(ac)
                sp=discretizador(nova,resolucao); retorno+=recompensa
                if algoritmo=="sarsa_lambda":
                    ap=escolher(q,sp,epsilon,rng) if not terminou else None
                    alvo=recompensa+(gamma*q[sp+(ap,)] if not terminou else 0)
                    delta=alvo-q[s+(ac,)]; traces[s+(ac,)]+=1
                    q += alpha*delta*traces; traces *= gamma*lam
                    ac=ap
                else:
                    alvo=recompensa+(gamma*np.max(q[sp]) if not terminou else 0)
                    q[s+(ac,)]+=alpha*(alvo-q[s+(ac,)])
                    if algoritmo=="dyna_q":
                        chave=(s,ac); modelo[chave][(float(recompensa),sp,bool(terminou))]+=1
                        chaves=list(modelo)
                        for _ in range(N):
                            sm,am=chaves[int(rng.integers(len(chaves)))]
                            contagem=modelo[(sm,am)]; desfechos=list(contagem)
                            pesos=np.array(list(contagem.values()),dtype=float); pesos/=pesos.sum()
                            rm,spm,tm=desfechos[int(rng.choice(len(desfechos),p=pesos))]
                            alvo=rm+(gamma*np.max(q[spm]) if not tm else 0)
                            q[sm+(am,)]+=alpha*(alvo-q[sm+(am,)])
                s=sp
                if terminou or truncou:
                    venceu=bool(terminou);break
            historico.append({"episodio":ep+1,"retorno":float(retorno),"sucesso":venceu,"passos":t+1})
            epsilon=max(.02,epsilon*decay)
    finally: env.close()
    return q, pd.DataFrame(historico)

RUN_DEMO = False  # altere para True para treinar agentes NOVOS, fora da triagem
if RUN_DEMO:
    historicos={}
    for algoritmo in ["q_learning","sarsa_lambda","dyna_q"]:
        _,h=treinar_demo(algoritmo,episodios=40,seed=100)
        historicos[algoritmo]=h
        print(algoritmo,"sucesso no treino curto:",f"{h.sucesso.mean():.1%}")
    fig,ax=plt.subplots(figsize=(9,4))
    for a,h in historicos.items():
        ax.plot(h.episodio,h.retorno.rolling(5,min_periods=1).mean(),label=nomes[a])
    ax.set_xlabel("Episódios do DEMO");ax.set_ylabel("Retorno suavizado")
    ax.legend(); ax.grid(alpha=.25); fig.tight_layout()
    fig.savefig(OUT/"demo_treino_novo.png",dpi=220); plt.show()

## 5. Salvar arquivos da apresentação

Execute a célula para baixar um ZIP com as figuras geradas **nesta execução**. Ao apresentar, mantenha a distinção entre dados de triagem e demonstração didática.

In [ ]:
import shutil
arquivo=shutil.make_archive("CMP629_graficos_Colab","zip",root_dir=OUT)
print("Gerado:",arquivo)
try:
    from google.colab import files
    files.download(arquivo)
except ImportError:
    print("Fora do Colab: arquivo disponível no diretório atual.")